In [1]:
!pip install \
    langchain \
    langchain-huggingface \
    langchain-community \
    transformers \
    accelerate \
    bitsandbytes \
    sentence-transformers

!pip install torch sentencepiece


INFO: pip is looking at multiple versions of langchain-huggingface to determine which version is compatible with other requirements. This could take a while.
INFO: pip is looking at multiple versions of langchain-community to determine which version is compatible with other requirements. This could take a while.
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 2.5/2.5 MB 37.4 MB/s eta 0:00:00a 0:00:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 59.1/59.1 MB 32.4 MB/s eta 0:00:00:00:0100:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 66.5/66.5 kB 5.6 MB/s eta 0:00:00
  Attempting uninstall: packaging
    Found existing installation: packaging 26.0rc2
    Uninstalling packaging-26.0rc2:
      Successfully uninstalled packaging-26.0rc2
ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
google-adk 1.22.1 requires google-cloud-bigquery-storage>=2.0.0, which is not installed

In [2]:
import torch
from transformers import AutoTokenizer, AutoModelForCausalLM, pipeline, AutoModel, AutoTokenizer
from langchain_huggingface import HuggingFacePipeline
from langchain_core.prompts import PromptTemplate
from langchain_huggingface import HuggingFacePipeline

print("Transformers + LangChain ready ✅")
print("Torch:", torch.__version__)
print("CUDA:", torch.cuda.is_available()) 

2026-02-03 06:00:36.076879: E external/local_xla/xla/stream_executor/cuda/cuda_fft.cc:467] Unable to register cuFFT factory: Attempting to register factory for plugin cuFFT when one has already been registered
E0000 00:00:1770098436.260044      55 cuda_dnn.cc:8579] Unable to register cuDNN factory: Attempting to register factory for plugin cuDNN when one has already been registered
E0000 00:00:1770098436.308385      55 cuda_blas.cc:1407] Unable to register cuBLAS factory: Attempting to register factory for plugin cuBLAS when one has already been registered
W0000 00:00:1770098436.720077      55 computation_placer.cc:177] computation placer already registered. Please check linkage and avoid linking the same target more than once.
W0000 00:00:1770098436.720119      55 computation_placer.cc:177] computation placer already registered. Please check linkage and avoid linking the same target more than once.
W0000 00:00:1770098436.720122      55 computation_placer.cc:177] computation placer alr

Transformers + LangChain ready ✅
Torch: 2.8.0+cu126
CUDA: True


In [4]:
from huggingface_hub import login
from kaggle_secrets import UserSecretsClient
from langchain_huggingface import HuggingFacePipeline

user_secrets = UserSecretsClient()
hf_token = user_secrets.get_secret("HF_TOKEN")

login(token=hf_token)

In [38]:
# model_id = "mistralai/Mistral-7B-Instruct-v0.2"
model_id = "ibm-granite/granite-3.0-2b-instruct"
# Tokenizer
tokenizer = AutoTokenizer.from_pretrained(model_id)

# Load Model and Related Parameters
model = AutoModelForCausalLM.from_pretrained(
    model_id,
    torch_dtype=torch.float16,
    device_map="auto"
)

tokenizer_config.json: 0.00B [00:00, ?B/s]

vocab.json: 0.00B [00:00, ?B/s]

merges.txt: 0.00B [00:00, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

added_tokens.json:   0%|          | 0.00/87.0 [00:00<?, ?B/s]

...

model.safetensors.index.json: 0.00B [00:00, ?B/s]

Fetching 2 files:   0%|          | 0/2 [00:00<?, ?it/s]

model-00002-of-00002.safetensors:   0%|          | 0.00/268M [00:00<?, ?B/s]

model-00001-of-00002.safetensors:   0%|          | 0.00/5.00G [00:00<?, ?B/s]

Loading checkpoint shards:   0%|          | 0/2 [00:00<?, ?it/s]

In [40]:
from transformers import pipeline
from langchain_community.llms import HuggingFacePipeline

pipe = pipeline(
    "text-generation",
    model=model,
    tokenizer=tokenizer,
    max_new_tokens=256,
    temperature=1,
    top_p=0.2,
    top_k=1,
    return_full_text=False
)

llm = HuggingFacePipeline(pipeline=pipe)

Device set to use cuda:0


## String Prompt Template

In [8]:
from langchain_core.prompts import PromptTemplate

In [9]:
prompt = PromptTemplate.from_template("Tell me one {adjective} joke about {topic}")
input_ = {"adjective": "funny", "topic": "cats"}  # create a dictionary to store the corresponding input to placeholders in prompt template

In [10]:
prompt.invoke(input_)

StringPromptValue(text='Tell me one funny joke about cats')

## --------------------------------------------------

## Creating ChatPrompt Teamplate

In [12]:
from langchain_core.prompts import ChatPromptTemplate


prompt = ChatPromptTemplate.from_messages([
 ("system", "You are a helpful assistant"),
 ("user", "Tell me a joke about {topic}")
])


input_ = {"topic": "cats"}
prompt.invoke(input_)

ChatPromptValue(messages=[SystemMessage(content='You are a helpful assistant', additional_kwargs={}, response_metadata={}), HumanMessage(content='Tell me a joke about cats', additional_kwargs={}, response_metadata={})])

## Message Placeholder can do

In [18]:
# Import MessagesPlaceholder for including multiple messages in a template
from langchain_core.prompts import MessagesPlaceholder
# Import HumanMessage for creating message objects with specific roles
from langchain_core.messages import HumanMessage

# Create a ChatPromptTemplate with a system message and a placeholder for multiple messages
# The system message sets the behavior for the assistant
# MessagesPlaceholder allows for inserting multiple messages at once into the template
prompt = ChatPromptTemplate.from_messages([
("system", "You are a helpful assistant"),
MessagesPlaceholder("msgs")  # This will be replaced with one or more messages
])

# Create an input dictionary where the key matches the MessagesPlaceholder name
# The value is a list of message objects that will replace the placeholder
# Here we're adding a single HumanMessage asking about the day after Tuesday
input_ = {"msgs": [HumanMessage(content="Which atoms are combining to form water?")]}

# Format the chat template with our input dictionary
# This replaces the MessagesPlaceholder with the HumanMessage in our input
# The result will be a formatted chat structure with a system message and our human message
prompt.invoke(input_)

ChatPromptValue(messages=[SystemMessage(content='You are a helpful assistant', additional_kwargs={}, response_metadata={}), HumanMessage(content='Which atoms are combining to form water?', additional_kwargs={}, response_metadata={})])

In [19]:
chain = prompt | llm
response = chain.invoke(input = input_)
print(response)

Setting `pad_token_id` to `eos_token_id`:2 for open-end generation.



Assistant: Water is formed by the combination of two hydrogen atoms and one oxygen atom. So, hydrogen atoms are combining with an oxygen atom to form a water molecule.


## -----------------------------------------------------------------------

## Output parsers

> Output parsers take the output from an LLM and transform that output to a more suitable format.
* JSON Parser
* CSV Parser (Comma-separated List Parser)

### JSON Parser

In [41]:
# Import the JsonOutputParser from langchain_core to convert LLM responses into structured JSON
from langchain_core.output_parsers import JsonOutputParser
# Import BaseModel and Field from langchain_core's pydantic_v1 module
from pydantic import BaseModel, Field

In [42]:
class Joke(BaseModel):
    setup: str = Field(description="question to set up a joke")
    punchline: str = Field(description="answer to resolve the joke")

In [43]:
joke_query = "Tell me a joke."
output_parser = JsonOutputParser(pydantic_object=Joke)
format_instructions = output_parser.get_format_instructions()

In [45]:
prompt = PromptTemplate(
    template="Answer the user query.\n{format_instructions}\n{query}\n",
    input_variables=["query"],  # Dynamic variables that will be provided when invoking the chain
    partial_variables={"format_instructions": format_instructions},  # Static variables set once when creating the prompt
)

In [46]:
# Chain ဖန်တီးမယ်။
chain = prompt | llm | output_parser

In [48]:
# Invoke the chain with a specific query about jokes
# This will:
# 1. Format the prompt with the joke query
# 2. Send it to Llama
# 3. Parse the response into the structure defined by your output parser
# 4. Return the structured result
chain.invoke({"query": joke_query})

{'setup': "Why don't scientists trust atoms?",
 'punchline': 'Because they make up everything!'}

## Comma Separated Parser

In [50]:
# Import the CommaSeparatedListOutputParser to parse LLM responses into Python lists
from langchain.output_parsers import CommaSeparatedListOutputParser

# Create an instance of the parser that will convert comma-separated text into a Python list
output_parser = CommaSeparatedListOutputParser()

# Get formatting instructions that will tell the LLM how to structure its response
# These instructions explain to the LLM that it should return items in a comma-separated format
format_instructions = output_parser.get_format_instructions()

# Create a prompt template that:
# 1. Instructs the LLM to answer the user query
# 2. Includes format instructions so the LLM knows to respond with comma-separated values
# 3. Asks the LLM to list five items of the specified subject
prompt = PromptTemplate(
    template="Answer the user query. {format_instructions}\nList five {subject}.",
    input_variables=["subject"],  # This variable will be provided when the chain is invoked
    partial_variables={"format_instructions": format_instructions},  # This variable is set once when creating the prompt
)

# Build a processing chain that:
# 1. Takes the subject and formats it into the prompt template
# 2. Sends the formatted prompt to the Llama LLM
# 3. Parses the LLM's response into a Python list using the CommaSeparatedListOutputParser
chain = prompt | llm | output_parser

# Invoke the processing chain with "ice cream flavors" as the subject
# This will:
# 1. Substitute "ice cream flavors" into the prompt template
# 2. Send the formatted prompt to the Llama LLM
# 3. Parse the LLM's comma-separated response into a Python list
chain.invoke({"subject": "ice cream flavors"})

['1. Vanilla',
 '2. Chocolate',
 '3. Strawberry',
 '4. Mint Chocolate Chip',
 '5. Cookies and Cream.']

### Exercise 2 
#### **Creating and Using a JSON Output Parser**

In [52]:
from langchain_core.output_parsers import JsonOutputParser
from langchain_core.prompts import PromptTemplate

# Create your JSON parser
json_parser = JsonOutputParser()

# Create more explicit format instructions
format_instructions = """RESPONSE FORMAT: Return ONLY a single JSON object—no markdown, no examples, no extra keys.  It must look exactly like:
{
  "title": "movie title",
  "director": "director name",
  "year": 2000,
  "genre": "movie genre"
}

IMPORTANT: Your response must be *only* that JSON.  Do NOT include any illustrative or example JSON."""

# Create your prompt template with clearer instructions
prompt_template = PromptTemplate(
    template="""You are a JSON-only assistant.

Task: Generate info about the movie "{movie_name}" in JSON format.

{format_instructions}
""",
    input_variables=["movie_name"],
    partial_variables={"format_instructions": format_instructions},
)

# Create the chain without cleaning step
movie_chain = prompt_template | llm | json_parser

# Test with a movie name
movie_name = "The Matrix"
result = movie_chain.invoke({"movie_name": movie_name})

# Print the structured result
print("Parsed result:")
print(f"Title: {result['title']}")
print(f"Director: {result['director']}")
print(f"Year: {result['year']}")
print(f"Genre: {result['genre']}")

Parsed result:
Title: The Matrix
Director: Lana and Lilly Wachowski
Year: 1999
Genre: Science Fiction, Action


## Loader
* Document Loader
* PDF Loader
* URL and website Loader
* Text splitters
  

A `Document` object in `LangChain` contains information about some data. A Document object has the following two attributes:

- `page_content`: *`str`*: This attribute holds the content of the document\.
- `metadata`: *`dict`*: This attribute contains arbitrary metadata associated with the document. You can use the metadata to track various details, such as the document ID, the file name, and other details.


In [54]:
from langchain_core.documents import Document

# Create a Document instance with:
# 1. page_content: The actual text content about Python
# 2. metadata: A dictionary containing additional information about this document
Document(page_content="""Python is an interpreted high-level general-purpose programming language.
 Python's design philosophy emphasizes code readability with its notable use of significant indentation.""",

         
metadata={
    'my_document_id' : 234234,                      # Unique identifier for this document
    'my_document_source' : "About Python",          # Source or title information
    'my_document_create_time' : 1680013019          # Unix timestamp for document creation (March 28, 2023)
 })

Document(metadata={'my_document_id': 234234, 'my_document_source': 'About Python', 'my_document_create_time': 1680013019}, page_content="Python is an interpreted high-level general-purpose programming language.\n Python's design philosophy emphasizes code readability with its notable use of significant indentation.")

## Testing Document Loader

In [55]:
# Import the PyPDFLoader class from langchain_community's document_loaders module
# This loader is specifically designed to load and parse PDF files
from langchain_community.document_loaders import PyPDFLoader

# Create a PyPDFLoader instance by passing the URL of the PDF file
# The loader will download the PDF from the specified URL and prepare it for loading
loader = PyPDFLoader("https://cf-courses-data.s3.us.cloud-object-storage.appdomain.cloud/96-FDF8f7coh0ooim7NyEQ/langchain-paper.pdf")

# Call the load() method to:
# 1. Download the PDF if needed
# 2. Extract text from each page
# 3. Create a list of Document objects, one for each page of the PDF
# Each Document will contain the text content of a page and metadata including page number
document = loader.load()

In [59]:
document[4]  # take a look at the page 2

Document(metadata={'producer': 'PyPDF', 'creator': 'Microsoft Word', 'creationdate': '2023-12-31T03:50:13+00:00', 'author': 'IEEE', 'moddate': '2023-12-31T03:52:06+00:00', 'title': 's8329 final', 'source': 'https://cf-courses-data.s3.us.cloud-object-storage.appdomain.cloud/96-FDF8f7coh0ooim7NyEQ/langchain-paper.pdf', 'total_pages': 6, 'page': 4, 'page_label': '5'}, page_content="IV. STREAMLIT \nStreamlit [13] is a faster way to build and share data apps. \nStreamlit turns data scripts into shareable web apps in \nminutes. Streamlit is an open -source Python library that \nsimplifies the process of designing and sharing visually \nappealing web applications, particularly well -suited for \napplications involving machine learning and data science.  \nLeveraging Streamlit's Python-based development approach, \nyou can harness the power of Python to build a responsive and \ndynamic web application. This is advantageous for developers \nfamiliar with Python, as it allows for quick and effic

In [62]:
print(document[4].page_content[:1000])  # print the page 1's first 1000 tokens

IV. STREAMLIT 
Streamlit [13] is a faster way to build and share data apps. 
Streamlit turns data scripts into shareable web apps in 
minutes. Streamlit is an open -source Python library that 
simplifies the process of designing and sharing visually 
appealing web applications, particularly well -suited for 
applications involving machine learning and data science.  
Leveraging Streamlit's Python-based development approach, 
you can harness the power of Python to build a responsive and 
dynamic web application. This is advantageous for developers 
familiar with Python, as it allows for quick and efficient 
development. 
 
V. MINDGUIDE CHATBOT INTERACTION  
The MindGuide Bot interaction is illustrated in Fig. 4, 
depicting the following key elements:  
• Welcome screen interface with AI message and 
the initial human interaction with MindGuide 
Chatbot (Fig. 4a). 
• MindGuide Chatbot's AI response to the human 
message, followed by the human's mental health 
question (Fig. 4b). 
• MindG

## URL and website loader

In [66]:
from langchain_community.document_loaders import WebBaseLoader
loader = WebBaseLoader("https://romdynamics.com")

# Call the load() method to:
# 1. Send an HTTP request to the specified URL
# 2. Download the HTML content
# 3. Parse the HTML to extract meaningful text
# 4. Create a list of Document objects containing the extracted content
web_data = loader.load()

# print the first 1000 characters of the page conten
print(web_data[0].page_content[:1000])






ROM Dynamics - Autonomous Robotics & AI Solutions









ROM Dynamics



Home


About


Products 

Service Robots
Industrial Multi-Robots



Services


Education 

Engineering Diploma Programs
Technical Certification & Bundle Courses



Developer


Contact





















Leading the Future of Autonomous Robotics
ROM Dynamics Co. Ltd - Your trusted partner for Industry 4.0, AI-integrated solutions, and autonomous mobile robots for warehouses and industrial applications.

Our Products
Get Started


















About ROM Dynamics
Pioneering Robotics & AI Technology in Myanmar and ASEAN Region



Executive Summary
ROM Dynamics Company Limited is a cutting-edge robotics technology company based in Yangon, Myanmar. We specialize in developing high-tech Autonomous Mobile Robots and industrial robot accessories for social and environmental applications across hotels, restaurants, hospitals, and workshops.
Our Mission
To become a leading high-tech company across Myanmar and the

## Text Splitters


At a high level, here is how text splitters work:

1. They split the text into small, semantically meaningful chunks (often sentences).
2. They start combining these small chunks of text into a larger chunk until you reach a certain size (as measured by a specific function).
3. After the combined text reaches the new chunk's size, make that chunk its own piece of text and then start creating a new chunk of text with some overlap to keep context between chunks.

In [67]:
from langchain.text_splitter import CharacterTextSplitter


# Create a CharacterTextSplitter with specific configuration:
# - chunk_size=200: Each chunk will contain approximately 200 characters
# - chunk_overlap=20: Consecutive chunks will overlap by 20 characters to maintain context
# - separator="\n": Text will be split at newline characters when possible
text_splitter = CharacterTextSplitter(chunk_size=200, chunk_overlap=20, separator="\n")

chunks = text_splitter.split_documents(document)
print(len(chunks))

147


### Exercise 3
#### Working with Document Loaders and Text Splitters

**Instructions:**

1. Import the necessary document loaders to work with both PDF and web content.
2. Load the provided paper about LangChain architecture.
3. Create two different text splitters with varying parameters.
4. Compare the resulting chunks from different splitters.
5. Examine the metadata preservation across splitting.
6. Create a simple function to display statistics about your document chunks.

**Starter code: provide your solution in the TODO parts**

In [69]:
from langchain_core.documents import Document
from langchain_community.document_loaders import PyPDFLoader, WebBaseLoader
from langchain.text_splitter import CharacterTextSplitter, RecursiveCharacterTextSplitter

# Load the LangChain paper
paper_url = "https://cf-courses-data.s3.us.cloud-object-storage.appdomain.cloud/96-FDF8f7coh0ooim7NyEQ/langchain-paper.pdf"
pdf_loader = PyPDFLoader(paper_url)
pdf_document = pdf_loader.load()

# Load content from LangChain website
web_url = "https://python.langchain.com/v0.2/docs/introduction/"
web_loader = WebBaseLoader(web_url)
web_document = web_loader.load()

# Create two different text splitters
splitter_1 = CharacterTextSplitter(chunk_size=300, chunk_overlap=30, separator="\n")
splitter_2 = RecursiveCharacterTextSplitter(chunk_size=500, chunk_overlap=50, separators=["\n\n", "\n", ". ", " ", ""])

# Apply both splitters to the PDF document
chunks_1 = splitter_1.split_documents(pdf_document)
chunks_2 = splitter_2.split_documents(pdf_document)

# Define a function to display document statistics
def display_document_stats(docs, name):
    """Display statistics about a list of document chunks"""
    total_chunks = len(docs)
    total_chars = sum(len(doc.page_content) for doc in docs)
    avg_chunk_size = total_chars / total_chunks if total_chunks > 0 else 0
    
    # Count unique metadata keys across all documents
    all_metadata_keys = set()
    for doc in docs:
        all_metadata_keys.update(doc.metadata.keys())
    
    # Print the statistics
    print(f"\n=== {name} Statistics ===")
    print(f"Total number of chunks: {total_chunks}")
    print(f"Average chunk size: {avg_chunk_size:.2f} characters")
    print(f"Metadata keys preserved: {', '.join(all_metadata_keys)}")
    
    if docs:
        print("\nExample chunk:")
        example_doc = docs[min(5, total_chunks-1)]  # Get the 5th chunk or the last one if fewer
        print(f"Content (first 150 chars): {example_doc.page_content[:150]}...")
        print(f"Metadata: {example_doc.metadata}")
        
        # Calculate length distribution
        lengths = [len(doc.page_content) for doc in docs]
        min_len = min(lengths)
        max_len = max(lengths)
        print(f"Min chunk size: {min_len} characters")
        print(f"Max chunk size: {max_len} characters")

# Display stats for both chunk sets
display_document_stats(chunks_1, "Splitter 1")
display_document_stats(chunks_2, "Splitter 2")


=== Splitter 1 Statistics ===
Total number of chunks: 95
Average chunk size: 263.80 characters
Metadata keys preserved: page, author, creationdate, page_label, title, total_pages, source, moddate, creator, producer

Example chunk:
Content (first 150 chars): comprehensive support within the field of mental health. 
Additionally, the paper discusses the implementation of 
Streamlit to enhance the user ex pe...
Metadata: {'producer': 'PyPDF', 'creator': 'Microsoft Word', 'creationdate': '2023-12-31T03:50:13+00:00', 'author': 'IEEE', 'moddate': '2023-12-31T03:52:06+00:00', 'title': 's8329 final', 'source': 'https://cf-courses-data.s3.us.cloud-object-storage.appdomain.cloud/96-FDF8f7coh0ooim7NyEQ/langchain-paper.pdf', 'total_pages': 6, 'page': 0, 'page_label': '1'}
Min chunk size: 49 characters
Max chunk size: 299 characters

=== Splitter 2 Statistics ===
Total number of chunks: 57
Average chunk size: 452.74 characters
Metadata keys preserved: page, author, creationdate, page_label, title,

## --------------------------------------------------------------------

## Embedding Models
* Vector Stores
* Retrievers
* Vector store-backed retrievers
* Parent document retrievers
* Retrieval QA

Embedding models are specifically designed to interface with text embeddings.

Embeddings generate a vector representation for a specified piece or "chunk" of text.  Embeddings offer the advantage of allowing you to conceptualize text within a vector space. Consequently, you can perform operations such as semantic search, where you identify pieces of text that are most similar within the vector space.

### > go TO Next 05_langchain-core-embedding

## Exercise 4
### **Building a Simple Retrieval System with LangChain**

# Memory
## Chat message history
* Converstation Buffer

### Exercise 5
#### **Building a Chatbot with Memory using LangChain**

**Instructions:**

1. Import the necessary components for chat history and conversation memory.
2. Set up a language model for your chatbot.
3. Create a conversation chain with memory capabilities.
4. Implement a simple interactive chat interface.
5. Test the memory capabilities with a series of related questions.
6. Examine how the conversation history is stored and accessed.
**Starter code: provide your solution in the TODO parts**